In [ ]:
import pickle as pk
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
)

SEED = 42
np.random.seed(SEED)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

# Week 3 Helper Notebook
This exploration should produce AT LEAST two figures , which you should include in the progress report and make look nice using seaborn (feel free to use AI to help with plotting). For example:
- A table or figure showing the most stable and unstable features.
- At least one comparison across feature categories.
- Different instability metrics. 


Guiding questions to discuss with your group: 

1. What are at least three reasonable definitions of feature instability?
2. Do those definitions identify the same features?
4. Are certain EHR feature categories systematically more unstable?
5. Does instability depend on the prediction outcome?
7. Which instability metrics seem most promising to investigate further?



## Step 1: Load eICU data

In [ ]:
feature_sets = pk.load(open('../../data/feature_names.pkl', 'rb'))
patient_agg = pd.read_parquet('../../data/clean_dataset.parquet')

print("Dataset shape:", patient_agg.shape)
patient_agg.head()

Dataset shape: (135778, 2167)


,patienthealthsystemstayid,gender,age,ethnicity,hospitalid,hospitaldischargeyear,mortality_at48h,hospital_numbedscategory,hospital_region,"admissiondx_ARDS-adult respiratory distress syndrome, non-cardiogenic pulmonary edema",admissiondx_Abdomen only trauma,admissiondx_Abdomen/extremity trauma,admissiondx_Abdomen/face trauma,admissiondx_Abdomen/multiple trauma,admissiondx_Abdomen/pelvis trauma,admissiondx_Abdomen/spinal trauma,admissiondx_Ablation or mapping of cardiac conduction pathway,"admissiondx_Abscess, neurologic","admissiondx_Abscess/infection-cranial, surgery for",admissiondx_Acid-base/electrolyte disturbance,admissiondx_Addisons disease,admissiondx_Adrenal neoplasm (including pheochromocytoma),admissiondx_Adrenalectomy,admissiondx_Alcohol withdrawal,admissiondx_Amputation (non-traumatic),admissiondx_Amyotrophic lateral sclerosis,admissiondx_Anaphylaxis,"admissiondx_Anastomosis, vascular",admissiondx_Anemia,"admissiondx_Aneurysm repair, ventricular","admissiondx_Aneurysm, abdominal aortic","admissiondx_Aneurysm, abdominal aortic; with dissection","admissiondx_Aneurysm, abdominal aortic; with rupture","admissiondx_Aneurysm, dissecting aortic","admissiondx_Aneurysm, thoracic aortic","admissiondx_Aneurysm, thoracic aortic; with dissection","admissiondx_Aneurysm, thoracic aortic; with rupture","admissiondx_Aneurysm/pseudoaneurysm, other","admissiondx_Aneurysms, repair of other (except ventricular)","admissiondx_Angina, stable (asymp or stable pattern of symptoms w/meds)","admissiondx_Angina, unstable (angina interferes w/quality of life or meds are tolerated poorly)",admissiondx_Aortic and Mitral valve replacement,admissiondx_Aortic valve replacement (isolated),"admissiondx_Apnea, sleep","admissiondx_Apnea-sleep; surgery for (i.e., UPPP - uvulopalatopharyngoplasty)",admissiondx_Appendectomy,"admissiondx_Arrest, respiratory (without cardiac arrest)","admissiondx_Arteriovenous malformation, surgery for","admissiondx_Arthritis, rheumatoid","admissiondx_Arthritis, septic",...,calcium,cd 4,chloride,cortisol,creatinine,direct bilirubin,eos,ethanol,fibrinogen,folate,free T4,glucose,glucose CSF,haptoglobin,ionized calcium,lactate,lipase,lymphs,magnesium,monos,myoglobin,pH,paCO2,paO2,phosphate,platelets x 1000,polys,potassium,prealbumin,prolactin,protein CSF,protein C,protein S,reticulocyte count,salicylate,serum ketones,serum osmolality,sodium,total bilirubin,total cholesterol,total protein,transferrin,triglycerides,troponin I,troponin T,uric acid,urinary creatinine,urinary osmolality,urinary sodium,urinary specific gravity
0,128919,Female,70.0,Caucasian,59,2015,1,<100,Midwest,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,9.000000,NaN,101.500000,NaN,2.125000,NaN,0.5,NaN,NaN,NaN,NaN,113.0,NaN,NaN,NaN,NaN,NaN,12.500000,NaN,16.500000,NaN,NaN,NaN,NaN,NaN,211.000000,70.5,4.100000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,139.000000,3.35,NaN,7.10,NaN,NaN,NaN,NaN,8.1,NaN,NaN,NaN,NaN
1,128927,Female,52.0,Caucasian,60,2015,0,<100,Midwest,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,8.100000,NaN,107.750000,NaN,0.700000,NaN,3.0,234.0,NaN,NaN,NaN,74.5,NaN,NaN,NaN,NaN,NaN,45.000000,1.90,7.000000,NaN,NaN,NaN,NaN,NaN,273.000000,45.0,3.750000,NaN,NaN,NaN,NaN,NaN,NaN,2.3,NaN,NaN,144.500000,0.40,NaN,7.40,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,128941,Male,68.0,Caucasian,73,2015,0,>= 500,Midwest,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,9.200000,NaN,103.500000,NaN,2.725000,0.1,0.5,NaN,NaN,NaN,NaN,151.5,NaN,NaN,NaN,1.666667,NaN,3.500000,1.20,4.500000,NaN,NaN,NaN,NaN,NaN,265.500000,91.5,4.300000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,134.500000,0.40,NaN,7.45,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.018
3,128943,Male,71.0,Caucasian,67,2015,0,None,Midwest,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,9.000000,NaN,98.000000,NaN,0.865000,NaN,0.0,NaN,NaN,NaN,NaN,153.0,NaN,NaN,NaN,0.800000,NaN,3.000000,NaN,4.500

In [ ]:
# This distinction of numerical (continuous), binary, and categorical
# columns may be useful throughout the project.
all_num_cols = (
    feature_sets['labs'].tolist()
    + feature_sets['vitals'].tolist()
    + ['age', 'admissionheight', 'admissionweight']
)

all_bin_cols = (
    feature_sets['icd10_before24h'].tolist()
    + feature_sets['admissiondx'].tolist()
)

all_cat_cols = [
    # 'hospital_region',
    'ethnicity',
    'gender',
    'hospital_numbedscategory',
    'hospitaldischargeyear',
    # 'hospitalid'
]

In [ ]:
# For privacy, eICU records patients older than 89 using the string "> 89".
if 'age' in patient_agg.columns:
    patient_agg.loc[patient_agg.age == '> 89', 'age'] = 90
    patient_agg['age'] = patient_agg['age'].astype(float)

## Step 2: Prediction setup

In [ ]:
# todo: use the splits, outcome, model, and metrics from the previous notebook to train and evaluate a model here
# you should have a set of $K$ hospitals in the training data, which, when applied to the unseen hospitals, leads to a
# large generalization gap . 

## Step 3: Define potential types / measurements of instability
There is no single obvious definition of an unstable feature. This week, experiment with several possibilities.
Think of : What do prior works do? Any other ideas? 

That is, for a feature $X_j$, consider at least the following:
1. Does the distribution of $X_j$ differ substantially across the $K$ training hospitals? How do we measure that for continuous vs. binary variables? (hint: use statistical tests like KS tests)
2. Does the probability that a variable is observed differ across hospitals?
3. Does the relationship between $X_j$ and the outcome  $Y$ differ across hospitals? And how would we measure that? 
4. .... your ideas here!! 





In [ ]:

# eg. 
def measure_distribution_instability(feature_name, group_column, dataframe):
    """
    Measures distribution instability of a feature across groups (e.g. hospital) in a dataframe.
    Separate into continuous and categorical options. Return statistic and pvalue. 
    """
    ....  




In [ ]:
# BONUS! You will do this next week, but you can add a "predictive utility" measurement for each variable
# for how well it predicts $Y$

## Step 4: Build table to summarize results

Using only your **training hospitals**, create a table with one row per feature, and one column for each type of instability metric. 

eg. 
```
| Feature | Feature Type | Distribution Instability | ... | 
|-------------------------------------------------------|
| age | demographic | ... | ... | ... | ... | ... |
| creatinine | lab | ... | ... | ... | ... | ... |
| ventilation | procedure | ... | ... | ... | ... | ... |```

In [ ]:
# First, build the table 
# Then, compare feature types ie. vitals, demographics, diagnoses .... 

In [ ]:
# Then test if some categories systematically appear more unstable than others. 
# For example: Are laboratory missingness patterns more variable across hospitals than the lab values themselves?

In [ ]:
#TODO: Create at least one visualization comparing instability across feature categories.

## Step 5: Compare across different outcomes $Y$ and hospital splits 

Repeat the analysis for at least:

- two prediction outcomes
- multiple reasonable hospital splits that produce heldout generalizaiton gap

Use this to determine whether your conclusions are specific to one particular setup. eg, ask yourself:
*  Are some feature categories stable for mortality but unstable for another outcome?
* Is a feature unstable across most hospital splits, or only for one unusual hospital?
* ...etc. 

Note: At this stage, this analysis is **exploratory**. Do not conclude that a particular type of instability causes poor generalization. But start generating hypotheses. 



In [ ]:
# TODO